# Week 01: Small Python Data-Processing App

**Objective:** Load a raw CSV file, clean it using reusable functions, and process numeric columns with NumPy.

This dataset represents student scores and attendance. It deliberately contains missing, non-numeric, whitespace-padded, and out-of-range values.

## 1. Environment and dependency

The script uses Python's built-in `csv` and `pathlib` modules plus NumPy. In a local virtual environment, install dependencies with `python -m pip install -r requirements.txt`. In Colab, run the next cell if needed.

In [1]:
%pip -q install 'numpy>=1.24,<3.0'
import csv
from pathlib import Path
import numpy as np
print('NumPy version:', np.__version__)

NumPy version: 2.3.5


## 2. Load the raw dataset

`load_dataset()` reads each CSV row as a dictionary. Keeping the original values as strings lets the cleaning function decide which values are valid.

In [2]:
from IPython.display import display
import pandas as pd

# If running locally, sample_data.csv should be in the same folder as this notebook.
data_path = Path('sample_data.csv')
if not data_path.exists():
    # Colab fallback: upload sample_data.csv when prompted.
    from google.colab import files
    uploaded = files.upload()
    data_path = Path('sample_data.csv')

with data_path.open('r', newline='', encoding='utf-8') as f:
    raw_rows = list(csv.DictReader(f))
print(f'Raw records loaded: {len(raw_rows)}')
display(raw_rows)

Raw records loaded: 8


[{'student': 'Amina', 'department': 'IT', 'score': '85', 'attendance': '92'}, {'student': 'Bekele', 'department': 'IT', 'score': '78', 'attendance': '88'}, {'student': 'Sara', 'department': 'Business', 'score': '91', 'attendance': '95'}, {'student': 'Dawit', 'department': 'IT', 'score': '', 'attendance': '81'}, {'student': 'Mekdes', 'department': 'Business', 'score': '67', 'attendance': '70'}, {'student': 'Robel', 'department': 'Business', 'score': '105', 'attendance': '89'}, {'student': 'Hana', 'department': 'IT', 'score': '  88 ', 'attendance': '97'}, {'student': 'Yonas', 'department': 'Business', 'score': 'absent', 'attendance': '76'}]

## 3. Clean the data with functions

The cleaning function trims whitespace, validates attendance, and converts scores to numbers. Missing, non-numeric, and out-of-range scores become `None`. Rows with unusable names, departments, or attendance are skipped.

In [3]:
def clean_dataset(rows):
    cleaned = []
    for row in rows:
        student = (row.get("student") or "").strip()
        department = (row.get("department") or "").strip()
        score_text = (row.get("score") or "").strip()
        attendance_text = (row.get("attendance") or "").strip()

        if not student or not department:
            continue

        try:
            attendance = float(attendance_text)
        except (TypeError, ValueError):
            continue
        if not 0 <= attendance <= 100:
            continue

        try:
            score = float(score_text)
        except (TypeError, ValueError):
            score = None

        if score is not None and not 0 <= score <= 100:
            score = None

        cleaned.append({
            "student": student,
            "department": department,
            "score": score,
            "attendance": attendance,
        })
    return cleaned

cleaned_rows = clean_dataset(raw_rows)
print(f"Records after cleaning: {len(cleaned_rows)}")
display(cleaned_rows)


Records after cleaning: 8


[{'student': 'Amina', 'department': 'IT', 'score': 85.0, 'attendance': 92.0}, {'student': 'Bekele', 'department': 'IT', 'score': 78.0, 'attendance': 88.0}, {'student': 'Sara', 'department': 'Business', 'score': 91.0, 'attendance': 95.0}, {'student': 'Dawit', 'department': 'IT', 'score': None, 'attendance': 81.0}, {'student': 'Mekdes', 'department': 'Business', 'score': 67.0, 'attendance': 70.0}, {'student': 'Robel', 'department': 'Business', 'score': None, 'attendance': 89.0}, {'student': 'Hana', 'department': 'IT', 'score': 88.0, 'attendance': 97.0}, {'student': 'Yonas', 'department': 'Business', 'score': None, 'attendance': 76.0}]

## 4. Process the data with NumPy

NumPy arrays make it straightforward to calculate descriptive statistics. Only valid scores are included in score statistics; attendance values are summarized for all retained records.

In [4]:
def process_dataset(rows):
    valid_scores = np.array(
        [row["score"] for row in rows if row["score"] is not None],
        dtype=float,
    )
    attendance = np.array([row["attendance"] for row in rows], dtype=float)

    return {
        "records_after_cleaning": len(rows),
        "valid_scores_count": int(valid_scores.size),
        "missing_or_invalid_scores": int(
            sum(row["score"] is None for row in rows)
        ),
        "mean_score": float(np.mean(valid_scores)) if valid_scores.size else None,
        "median_score": float(np.median(valid_scores)) if valid_scores.size else None,
        "minimum_score": float(np.min(valid_scores)) if valid_scores.size else None,
        "maximum_score": float(np.max(valid_scores)) if valid_scores.size else None,
        "mean_attendance": float(np.mean(attendance)) if attendance.size else None,
    }

summary = process_dataset(cleaned_rows)
for label, value in summary.items():
    print(f"{label.replace('_', ' ').title()}: {value:.2f}" if isinstance(value, float)
          else f"{label.replace('_', ' ').title()}: {value}")


Records After Cleaning: 8
Valid Scores Count: 5
Missing Or Invalid Scores: 3
Mean Score: 81.80
Median Score: 85.00
Minimum Score: 67.00
Maximum Score: 91.00
Mean Attendance: 86.00


## 5. Key observations

- The raw file contains a missing score, a non-numeric score, and a score above 100.
- Invalid scores are excluded from score statistics rather than being treated as zero.
- Attendance is processed separately, so a record can still contribute attendance when its score is invalid.
- The same cleaning and processing logic can be reused for another CSV file with the same column names.

## 6. Run the standalone Python script

From the `week-01` folder, install dependencies and execute:

```bash
python -m venv .venv
# Windows: .venv\\Scripts\\activate
# macOS/Linux: source .venv/bin/activate
python -m pip install -r requirements.txt
python data_processing_app.py
```

The `.py` file is the standalone source; this notebook is the documented, executed walkthrough.